# 02 Published detectors on the 69 pools
Every published detector we ran, scored on the same 69 held-out pools as our method: union AP and E1 (the comparison table of the paper), the metrics the tool papers use (precision, recall, F1, MCC, kappa, binary ARI/AMI of a call, and ARI/AMI of a partition against the hidden subtypes) and the exact atlas-level tests against our method.

Inputs: `pools.tgz`, `predictions.tgz` (our method and the in-house baselines) and `tool_predictions.tgz` (the saved output of each tool). Tools were run with the authors' code at default settings. A tool "with labels" is the tool's output post-processed with the visible labels. For FiRE the table row is the within-label percentile of the raw score (`fire_lab_pct`); `fire_lab` is the raw-score version and is shown for the stored analysis only. Nothing is rerun; scGAD is left out because it ran on only 32 of the pools.

K_* metrics call the top K cells (K = number of hidden cells; seeded tie-break), so precision, recall and F1 coincide. own_* metrics use the tool's own declared calls, and exist only for tools that declare calls. part_* metrics score a tool's partition against the hidden subtypes.

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
REPO = os.path.abspath(os.environ.get("REPO", ".."))
ASSETS = os.path.abspath(os.environ.get("ASSETS", os.path.join(REPO, "assets")))
sys.path.insert(0, REPO); sys.dont_write_bytecode = True
import json, numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
EXP = f"{REPO}/eval/expected"
from eval import paper as P
pids, man = P.pools(ASSETS)
M = {m: f"{ASSETS}/predictions/{m}" for m in ["hiddenrare", "leiden_sub", "iforest_pca", "lof_pca", "iforest_scvi"]}
M |= {m: f"{ASSETS}/tool_out/{m}" for m in ["sccad", "sccad_lab", "giniclust3", "giniclust3_lab", "fire", "fire_lab", "fire_lab_pct", "annogcd"]}
LM = P.literature_metrics(ASSETS, M, pids)
assert LM.groupby("method").pilot_id.nunique().eq(69).all()
LM["atlas"] = man.loc[LM.pilot_id, "atlas"].values
print(len(M), "methods,", LM.metric.nunique(), "metrics")

13 methods, 21 metrics


## 1. Atlas-equal means
Mean over atlases of the per-atlas mean of each metric. Compared with the stored values of the analysis (`eval/expected/atlas_equal_means.csv`) to 1e-9 for every method it covers (all but `fire_lab_pct`).

In [2]:
AE = LM.groupby(["method", "metric", "atlas"]).value.mean().groupby(["method", "metric"]).mean().unstack(0)
exp = pd.read_csv(f"{EXP}/atlas_equal_means.csv", index_col=0)
assert set(AE.columns) - set(exp.columns) == {"fire_lab_pct"} and set(AE.index) == set(exp.index)
S, X = AE[exp.columns], exp.loc[AE.index]
assert (S.isna() == X.isna()).all().all() and np.allclose(S.fillna(0), X.fillna(0), atol=1e-9), "atlas-equal means differ from the stored analysis"
order = ["hiddenrare", "sccad_lab", "leiden_sub", "iforest_scvi", "giniclust3_lab", "sccad", "iforest_pca", "fire_lab_pct", "giniclust3", "fire", "annogcd"]
print(AE.loc[["union_AP", "E1_union"], order].T.round(3))

metric          union_AP  E1_union
method                            
hiddenrare         0.321     0.323
sccad_lab          0.231     0.244
leiden_sub         0.220     0.234
iforest_scvi       0.177     0.218
giniclust3_lab     0.170     0.188
sccad              0.161     0.175
iforest_pca        0.150     0.183
fire_lab_pct       0.098     0.124
giniclust3         0.088     0.081
fire               0.070     0.066
annogcd            0.066     0.064


## 2. The comparison table
Union AP and E1 as printed in the comparison table of the paper (three decimals, same row order).

In [3]:
TABLE = {"hiddenrare": (.321, .323), "sccad_lab": (.231, .244), "leiden_sub": (.220, .234), "iforest_scvi": (.177, .218), "giniclust3_lab": (.170, .188),
         "sccad": (.161, .175), "iforest_pca": (.150, .183), "fire_lab_pct": (.098, .124), "giniclust3": (.088, .081), "fire": (.070, .066), "annogcd": (.066, .064)}
for m, (u, e) in TABLE.items():
    assert (round(AE.loc["union_AP", m], 3), round(AE.loc["E1_union", m], 3)) == (u, e), m
assert max(AE.loc["union_AP", order[1:]]) < AE.loc["union_AP", "hiddenrare"] and max(AE.loc["E1_union", order[1:]]) < AE.loc["E1_union", "hiddenrare"]
print("all 22 table entries reproduce; our method is highest on both metrics")

all 22 table entries reproduce; our method is highest on both metrics


## 3. Our method against every other method
Exact atlas-level sign-flip (2^8 patterns) on each metric, compared with the stored analysis (`eval/expected/vs_hiddenrare.csv`).

In [4]:
V = P.versus_table(LM, man)
E = pd.read_csv(f"{EXP}/vs_hiddenrare.csv")
J = V[V["vs"].isin(exp.columns)].merge(E, on=["metric", "vs"], suffixes=("", "_exp"), how="outer", indicator=True)
assert (J._merge == "both").all(), J[J._merge != "both"][["metric", "vs"]]
assert (J.n_pools == J.n_pools_exp).all() and (J.atlas_ahead == J.atlas_ahead_exp).all()
for c in ("ours_atlas_equal", "other_atlas_equal", "diff"): assert (J[c] - J[c + "_exp"]).abs().max() < 6e-5, c
assert (J.atlas_exact_p - J.atlas_exact_p_exp).abs().max() < 1e-9
print(len(J), "comparisons match")
U = V[V.metric == "union_AP"].set_index("vs"); print(U.loc[order[1:], ["n_pools", "other_atlas_equal", "diff", "atlas_ahead", "atlas_exact_p"]].round(4))

110 comparisons match
                n_pools  other_atlas_equal    diff  atlas_ahead  atlas_exact_p
vs                                                                            
sccad_lab            69             0.2309  0.0904            6         0.1406
leiden_sub           69             0.2204  0.1009            6         0.0312
iforest_scvi         69             0.1767  0.1446            7         0.0156
giniclust3_lab       69             0.1700  0.1513            7         0.0156
sccad                69             0.1607  0.1606            7         0.0312
iforest_pca          69             0.1501  0.1712            7         0.0156
fire_lab_pct         69             0.0975  0.2238            8         0.0078
giniclust3           69             0.0882  0.2331            8         0.0078
fire                 69             0.0700  0.2514            8         0.0078
annogcd              69             0.0656  0.2558            8         0.0078


Statements of the paper about individual tools.

In [5]:
for tool, diff, ahead, p in (("sccad", .161, 7, .031), ("giniclust3", .233, 8, .0078), ("sccad_lab", .090, 6, .14)):
    r = U.loc[tool]
    assert round(r["diff"], 3) == diff and r.atlas_ahead == ahead and round(r.atlas_exact_p, len(str(p).split(".")[1])) == p, (tool, r.to_dict())
    print(f"{tool}: +{r['diff']:.3f} union AP, ahead in {r.atlas_ahead} of 8 atlases, exact p = {r.atlas_exact_p:.4f}")

sccad: +0.161 union AP, ahead in 7 of 8 atlases, exact p = 0.0312
giniclust3: +0.233 union AP, ahead in 8 of 8 atlases, exact p = 0.0078
sccad_lab: +0.090 union AP, ahead in 6 of 8 atlases, exact p = 0.1406


## 4. Literature metrics
Atlas-equal means of the metrics of the tool papers. Our method declares no calls, so it has no own_* row.

In [6]:
show = ["K_F1", "K_MCC", "K_kappa", "K_ARI_bin", "own_P", "own_R", "own_F1", "own_MCC", "part_ARI_vs_finetype", "part_AMI_vs_finetype"]
print(AE.loc[show, order].round(3).to_string())

method                hiddenrare  sccad_lab  leiden_sub  iforest_scvi  giniclust3_lab  sccad  iforest_pca  fire_lab_pct  giniclust3   fire  annogcd
metric                                                                                                                                             
K_F1                       0.323      0.244       0.234         0.218           0.189  0.176        0.183         0.124       0.081  0.066    0.064
K_MCC                      0.290      0.206       0.197         0.179           0.150  0.134        0.143         0.082       0.037  0.019    0.017
K_kappa                    0.290      0.206       0.197         0.179           0.150  0.134        0.143         0.082       0.037  0.019    0.017
K_ARI_bin                  0.274      0.193       0.183         0.166           0.139  0.126        0.131         0.074       0.034  0.019    0.017
own_P                        NaN      0.347       0.235         0.148           0.112  0.141        0.166       

Every assertion compares a recomputed value with a stored value; the tools were not rerun here.